In [2]:
import pandas as pd 
import geopandas as gpd
import glob
import os


Faça o tratamento dos dfs de vcs definindo uma funcao aqui embaixo, e depois depois adicionem a funcao no mapa_tratamento falando o nome do arquivo correspondente. Fiz o tratar_ids como exemplo. Acho que assim vai ficar mais organizado

In [3]:
def tratar_ids(df):
    return df[['codbairro', 'bairro', 'ids']]

def obter_matriz_linha_bairo(df_routes, df_stops, df_route_stops):
    df_routes = df_routes.drop_duplicates(subset=['route_id', 'versao_modelo'])
    df_routes = df_routes[df_routes['route_type'] != 702]
    df_routes = df_routes[~df_routes['route_short_name'].str.contains('LECD|ESP01', na=False)]
    df_routes['route_short_name'] = df_routes['route_short_name'].str.lstrip('0')

    df_stops = df_stops.drop_duplicates(subset=['stop_id', 'versao_modelo'])

    df_route_stops = df_route_stops[df_route_stops['service_id'].isin(['D', 'D_REG', 'S', 'S_REG', 'U', 'U_REG'])]
    df_route_stops = df_route_stops.drop_duplicates(['stop_id', 'route_id', 'versao_modelo']).drop(['service_id'], axis=1)

    df_pipeline = pd.merge(
        df_route_stops, 
        df_routes[['route_id', 'versao_modelo', 'route_short_name']], 
        on=['route_id', 'versao_modelo'], 
        how='inner'
    )

    df_pipeline = pd.merge(
        df_pipeline, 
        df_stops[['stop_id', 'versao_modelo', 'stop_lat', 'stop_lon']], 
        on=['stop_id', 'versao_modelo'], 
        how='inner'
    )

    df_pipeline = df_pipeline.drop_duplicates().drop(['route_id', 'stop_id', 'versao_modelo'], axis=1).rename(columns={'route_short_name': 'linha', 'stop_lat': 'latitude', 'stop_lon': 'longitude'})

    gdf_onibus = gpd.GeoDataFrame(
        df_pipeline,
        geometry=gpd.points_from_xy(df_pipeline.longitude, df_pipeline.latitude),
        crs="EPSG:4326"
    )

    gdf_bairros = gpd.read_file('../dados/Limite_de_Bairros.geojson')

    gdf_bairros = gdf_bairros.to_crs("EPSG:4326")

    resultado = gpd.sjoin(gdf_onibus, gdf_bairros, how="inner", predicate="intersects")

    matriz = (
        resultado[['nome', 'linha']]
        .drop_duplicates()
        .rename(columns={'nome': 'bairro'})
        .sort_values(['bairro', 'linha'])
        .reset_index(drop=True)
    )

    return matriz



mapa_tratamento = {
    'ids_por_bairro_rj.csv': tratar_ids,
    # coloquem os nomes de cada arquivo e as funcoes correspondentes aqui
}

datasets = {}

# daqui em diante ele vai tratar tudo e colocar no dicionário datasets criado ali em cima
arquivos = glob.glob('../dados/*.csv')

for caminho in arquivos:
    nome_arquivo = os.path.basename(caminho)
    
    nome_df = os.path.splitext(nome_arquivo)[0].lower()
    
    df = pd.read_csv(caminho)
    
    if nome_arquivo in mapa_tratamento:
        df = mapa_tratamento[nome_arquivo](df)
    
    datasets[nome_df] = df

#colocando no dicionário 

df_routes = pd.read_csv('../dados/gtfs_novo/routes.csv')
df_stops = pd.read_csv('../dados/gtfs_novo/stops.csv')
df_route_stops = pd.read_csv('../dados/gtfs_novo/route_stops.csv')
datasets['matriz_linha_bairro'] = obter_matriz_linha_bairo(df_routes, df_stops, df_route_stops)
%store datasets


Stored 'datasets' (dict)


In [ ]:
#para acessar o dataframe em outros notebooks, você precisar executar %store -r datasets em alguma célula do notebook desejado (obs, tem que dar run all primeiro).
# pra acessar um dataframe específico, só fazer datasets['nome do arquivo sem a extensão']. vou deixar aq um exemplo
datasets['ids_por_bairro_rj'].head(5)

,codbairro,bairro,ids
0,13,Paquetá,0.571297
1,98,Freguesia (Ilha),0.594303
2,97,Bancários,0.589105
3,104,Galeão,0.562762
4,101,Tauá,0.576847


,bairro,linha
0,Abolição,2345
1,Abolição,254
2,Abolição,265
3,Abolição,371
4,Abolição,457
